# Fase 3 · Núcleo algorítmico, eficiencia y POO
## Licitaciones públicas de ChileCompra · MCDI500 · Grupo 2

F3 organiza las reglas del preprocesamiento revisado de F2 en un pipeline, contrasta sus dos salidas con las exportaciones de F2 y mide dos implementaciones del control de fechas. La unidad de análisis de la brecha es la licitación. La base de ofertas conserva el detalle por oferta e ítem.

**Cómo ejecutar.** Guarda este notebook en `F3/` y el módulo `pipeline_licitaciones_f3.py` en `src/`, junto con `funciones_f2_revision.py` y `pruebas_f2.py`. Mantén el CSV local en `data/raw/04LicitacionesEne.csv`. Ejecuta primero F2 con ese mismo archivo para generar ambas referencias en `data/processed/`. Abre F3 en el mismo entorno de Python y ejecuta todas sus celdas.

## 1. Configurar el entorno y ejecutar las pruebas de F2

In [1]:
from io import StringIO
from pathlib import Path
import platform
import sys
from timeit import repeat
import tracemalloc

import numpy as np
import pandas as pd
from IPython.display import display

inicio = Path.cwd()
raiz = next((p for p in (inicio, *inicio.parents) if (p / "src/pipeline_licitaciones_f3.py").is_file()), None)
if raiz is None:
    raise FileNotFoundError("Coloca el módulo de F3 en src/ y abre el notebook desde el proyecto.")
sys.path.insert(0, str(raiz))

from src.funciones_f2_revision import FECHAS, MONTOS, ORDEN_TAMANO
from src.pipeline_licitaciones_f3 import (
    Bitacora, ExtremosLogIQR, construir_pipeline, control_iterativo, control_vectorizado,
)
from src.pruebas_f2 import ejecutar_pruebas_f2

entrada = raiz / "data/raw/04LicitacionesEne.csv"
destino = raiz / "data/processed"
referencias = {
    "ofertas": destino / "04LicitacionesEne_procesado.csv",
    "licitaciones": destino / "04LicitacionesEne_brecha_por_licitacion.csv",
}
for ruta in [entrada, *referencias.values()]:
    if not ruta.is_file():
        raise FileNotFoundError(f"Falta {ruta.relative_to(raiz)}. Verifica la entrada y ejecuta F2 primero.")
print(f"Python {platform.python_version()} | pandas {pd.__version__} | NumPy {np.__version__}")
ejecutar_pruebas_f2()
print("Pruebas de las reglas de F2 aprobadas.")

Python 3.12.14 | pandas 2.2.3 | NumPy 2.3.5
Pruebas de las reglas de F2 aprobadas.


## 2. Ejecutar el pipeline de F3 y registrar los pasos

- **Herencia.** `TransformacionFija` y `ExtremosLogIQR` implementan el contrato `Transformador.transformar`. Las reglas fijas reutilizan las funciones ya probadas en F2.
- **Encapsulamiento.** `ExtremosLogIQR.ajustar` guarda los límites en `_limites`. `transformar` exige un ajuste previo y vuelve a usar esos límites.
- **Polimorfismo.** El ciclo del pipeline aplica las reglas fijas mediante `paso.transformar(datos)`. Cada objeto aplica la función que recibió. El IQR tiene un ajuste explícito antes de aplicar esa misma interfaz.
- **Observer.** `pipeline.suscribir(bitacora.registrar)` añade un observador que recibe los conteos después de cada paso. El pipeline puede ejecutarse sin observadores.
- **División funcional.** Usamos funciones para las reglas y un recorrido iterativo para encadenarlas. Las tablas tienen una secuencia de pasos fija.

La prueba del detector ajusta con montos entre 10 y 50 y evalúa una oferta de 1.000. Esto verifica que al transformar se van a usar los límites calculados con los datos de ajuste.

In [2]:
ajuste = pd.DataFrame({"MonedaOferta": ["CLP"] * 5, "MontoTotalOferta": [10, 20, 30, 40, 50]})
evaluacion = pd.DataFrame({"MonedaOferta": ["CLP", "USD"], "MontoTotalOferta": [1000, 1000]})
detector = ExtremosLogIQR()
detector.ajustar(ajuste)
bandera = detector.transformar(evaluacion)["EsExtremoLogIQR_CLP"]
pd.testing.assert_series_equal(
    bandera, pd.Series([True, pd.NA], dtype="boolean", name="EsExtremoLogIQR_CLP")
)
print("Se comprobó que la aplicación reutiliza los límites aprendidos.")

pipeline = construir_pipeline()
bitacora = Bitacora()
pipeline.suscribir(bitacora.registrar)
resultado = pipeline.ejecutar(entrada)
display(bitacora.tabla())
comparables = resultado.licitaciones["BrechaMonto"].notna()
print(f"Brecha disponible: {comparables.sum():,} de {len(comparables):,} licitaciones.")
print(f"Coincidencias con el monto estimado: {resultado.licitaciones['CoincideConEstimado'].sum():,}")

                             Paso  Filas de entrada  Filas de salida
0                      Cargar CSV             45741            45741
1   Marcar duplicados potenciales             45741            45741
2       Convertir fechas y montos             45741            45741
3            Normalizar faltantes             45741            45741
4     Ordenar tamaño de proveedor             45741            45741
5  Calcular brecha por licitación             45741             2292
6     Marcar valores extremos CLP             45741            45741
7                 Validar ofertas             45741            45741
8            Validar licitaciones              2292             2292

Se comprobó que la aplicación reutiliza los límites aprendidos.
Brecha disponible: 2,257 de 2,292 licitaciones.
Coincidencias con el monto estimado: 1,254


## 3. Comprobar equivalencia con las dos salidas reales de F2

Comparamos cada tabla de F3 con su CSV de referencia generado por F2. Para ambos lados leemos el formato CSV como texto. Así comparamos el orden, las columnas y todos los valores exportados con la misma representación. El CSV no preserva el tipo categórico de pandas. El pipeline comprueba ese tipo en memoria y el diccionario de F3 documenta el orden para recuperarlo.

In [3]:
tablas = {"ofertas": resultado.ofertas, "licitaciones": resultado.licitaciones}
for nombre, tabla in tablas.items():
    referencia = pd.read_csv(
        referencias[nombre], sep=";", encoding="utf-8-sig", dtype="string", keep_default_na=False
    )
    exportada = pd.read_csv(
        StringIO(tabla.to_csv(sep=";", index=False)), sep=";", dtype="string", keep_default_na=False
    )
    pd.testing.assert_frame_equal(referencia, exportada, check_exact=True)
    print(f"Equivalencia F2/F3 en {nombre}: {len(tabla):,} filas y {len(tabla.columns)} columnas.")

Equivalencia F2/F3 en ofertas: 45,741 filas y 17 columnas.
Equivalencia F2/F3 en licitaciones: 2,292 filas y 24 columnas.


## 4. Medir el control temporal

El control marca un cierre anterior a la publicación o una adjudicación anterior al cierre. Una comparación con una fecha faltante no activa la alerta. Comprobamos casos conocidos y exigimos que las dos implementaciones marquen las mismas filas en la base completa.

Las dos versiones tienen tiempo O(n). Medimos siete ejecuciones y mostramos el menor tiempo. Medimos por separado el pico de asignaciones rastreadas por `tracemalloc`; esa cifra puede omitir memoria nativa y no representa la memoria total del proceso. Los resultados dependen del equipo.

In [4]:
casos = pd.DataFrame([
    ["2025-01-01", "2025-01-03", "2025-01-05"],
    ["2025-01-03", "2025-01-02", None],
    [None, "2025-01-05", "2025-01-04"],
    ["2025-01-01", None, "2025-01-05"],
], columns=FECHAS).apply(pd.to_datetime)
esperado = pd.Series([False, True, True, False])
pd.testing.assert_series_equal(control_iterativo(casos), esperado)
pd.testing.assert_series_equal(control_vectorizado(casos), esperado)

ofertas = resultado.ofertas
alertas = control_vectorizado(ofertas)
pd.testing.assert_series_equal(control_iterativo(ofertas), alertas)

mediciones = []
for nombre, operacion in [
    ("Iterativa", control_iterativo), ("Vectorizada", control_vectorizado)
]:
    tiempos = repeat(lambda: operacion(ofertas), number=1, repeat=7)
    tracemalloc.start()
    try:
        operacion(ofertas)
        _, pico = tracemalloc.get_traced_memory()
    finally:
        tracemalloc.stop()
    mediciones.append({
        "Implementación": nombre, "Filas": len(ofertas), "Repeticiones": len(tiempos),
        "Tiempo mínimo (ms)": min(tiempos) * 1000, "Pico rastreado (KiB)": pico / 1024,
    })

tiempos = pd.DataFrame(mediciones)
display(tiempos.round(3))
razon = tiempos.loc[0, "Tiempo mínimo (ms)"] / tiempos.loc[1, "Tiempo mínimo (ms)"]
print(f"Razón de tiempos iterativa/vectorizada: {razon:.2f}×")
print(f"Alertas temporales: {alertas.sum():,}. Las dos implementaciones coinciden.")

  Implementación  Filas  Repeticiones  Tiempo mínimo (ms)  Pico rastreado (KiB)
0      Iterativa  45741             7              58.771              6318.651
1    Vectorizada  45741             7               0.141               225.470

Razón de tiempos iterativa/vectorizada: 416.99×
Alertas temporales: 0. Las dos implementaciones coinciden.


## 5. Preparar los insumos para Fase 4

El estudio es descriptivo y usa el conjunto completo. Se entregan las dos unidades verificadas con F2 y un diccionario con tipos y reglas. Los montos de monedas distintas se resumen por separado. La brecha expresa la diferencia entre montos registrados. Para interpretarla como desviación respecto de un presupuesto previo se necesitaría verificar cuándo se registró el estimado.

In [5]:
reglas = {
    **dict.fromkeys(FECHAS, "Conversión de fecha y hora desde el CSV"),
    **dict.fromkeys(MONTOS, "Conversión numérica con coma decimal"),
    "TamanoProveedorOrdinal": "Orden: " + " | ".join(ORDEN_TAMANO),
    "EsDuplicadoPotencial": "Coincidencia exacta entre filas tras excluir identificadores",
    "EsExtremoLogIQR_CLP": "IQR sobre logaritmos de montos positivos en CLP",
    "MontoTotalGanadoras": "Suma de montos ganadores en licitaciones comparables",
    "BrechaMonto": "Monto total ganador menos monto estimado registrado",
    "EstadoBrecha": "Estado de comparación o motivo principal de exclusión",
    "CoincideConEstimado": "Brecha absoluta menor o igual a 0,00000001",
}
diccionario = pd.DataFrame([
    {"Tabla": nombre, "Columna": columna, "Tipo en Python": str(tabla[columna].dtype),
     "Unidad de análisis": "Oferta por ítem" if nombre == "ofertas" else "Licitación",
     "Regla": reglas.get(columna, "Campo de entrada o control de la agregación de F2")}
    for nombre, tabla in tablas.items() for columna in tabla
])
parametros = pd.DataFrame([
    ("Tamaño ordinal", "categorías", " | ".join(ORDEN_TAMANO), "Fijo"),
    ("Brecha", "tolerancia absoluta de coincidencia", 1e-8, "Fijo"),
    ("IQR CLP", "multiplicador", 1.5, "Fijo"),
    *[("IQR CLP", clave, valor, "Ajustado sobre montos positivos en CLP")
      for clave, valor in (resultado.limites_iqr or {}).items()],
], columns=["Paso", "Parámetro", "Valor", "Origen"])

for nombre, tabla in {
    "ofertas_f3.csv": ofertas, "licitaciones_f3.csv": resultado.licitaciones,
    "diccionario_f3.csv": diccionario, "parametros_f3.csv": parametros,
}.items():
    ruta = destino / nombre
    temporal = ruta.with_suffix(".csv.tmp")
    tabla.to_csv(temporal, sep=";", encoding="utf-8-sig", index=False)
    temporal.replace(ruta)
    print(f"{ruta.relative_to(raiz)} · {len(tabla):,} filas")
display(parametros)

             Paso                            Parámetro                               Valor                                  Origen
0  Tamaño ordinal                           categorías  Micro | Pequeña | Mediana | Grande                                    Fijo
1          Brecha  tolerancia absoluta de coincidencia                                 0.0                                    Fijo
2         IQR CLP                        multiplicador                                 1.5                                    Fijo
3         IQR CLP                      ofertas_validas                               45603  Ajustado sobre montos positivos en CLP
4         IQR CLP                               q1_log                           10.683179  Ajustado sobre montos positivos en CLP
5         IQR CLP                               q3_log                           14.404324  Ajustado sobre montos positivos en CLP
6         IQR CLP                         inferior_log                            5

data/processed/ofertas_f3.csv · 45,741 filas
data/processed/licitaciones_f3.csv · 2,292 filas
data/processed/diccionario_f3.csv · 41 filas
data/processed/parametros_f3.csv · 8 filas


## 6. Verificar las cifras solicitadas por el profesor

Esta revisión usa el CSV local de trabajo. Los conteos se recalculan al ejecutar el notebook. El cuadro de faltantes muestra el tratamiento previsto para las columnas analíticas y los campos con más del 90 % de ausencia. El control de duplicados usa las columnas que quedaron tras excluir los identificadores directos. Conservamos las filas coincidentes y comparamos el resultado al excluir de la comparación las licitaciones con ganadoras potencialmente duplicadas.

La coincidencia se reporta con igualdad numérica exacta y con la tolerancia de F2. La cifra de 1.263 coincidencias indicada por el profesor sirve para contrastar, dado que revisó el CSV original con su propia implementación. El diccionario de la API describe la visibilidad y el monto estimado. Los indicadores derivados de la base y la justificación del estimado ayudan a revisar su origen, pero el momento de registro sigue sin verificarse. La pregunta del proyecto se formula como una comparación descriptiva de los montos registrados.

In [6]:
from src.funciones_f2_revision import (
    MOTIVOS, cargar_csv, convertir_tipos, marcar_duplicados, normalizar_codigo,
    tratar_faltantes,
)

entrada_auditada = cargar_csv(entrada)
nulos_al_leer = entrada_auditada.isna().sum()
marcadas, duplicados = marcar_duplicados(entrada_auditada)
convertidas, conversiones = convertir_tipos(marcadas)
_, faltantes = tratar_faltantes(convertidas, nulos_al_leer.reindex(convertidas.columns))
faltantes = faltantes.drop(index="EsDuplicadoPotencial")
print(f"Columnas de entrada tras el filtro de identificadores: {entrada_auditada.shape[1]}")
print(f"Filas adicionales idénticas: {int(duplicados['FilasAdicionales'].sum()):,}")
display(conversiones)
display(faltantes.loc[faltantes.index.isin(MOTIVOS) | faltantes["% faltantes"].gt(90)])

brechas = resultado.licitaciones
validas = brechas.loc[brechas["EstadoBrecha"].eq("Calculada")]
sin_duplicadas = validas.loc[~validas["GanadoraDuplicada"]]
print(f"Cobertura por licitación: {len(validas):,} de {len(brechas):,} ({len(validas) / len(brechas):.1%})")
for etiqueta, muestra in [("Todas las comparables", validas), ("Sensibilidad sin ganadoras duplicadas", sin_duplicadas)]:
    print(etiqueta)
    print(f"Licitaciones: {len(muestra):,}")
    print(f"Coincidencias exactas: {muestra['BrechaMonto'].eq(0).sum():,}")
    print(f"Coincidencias con tolerancia F2: {muestra['CoincideConEstimado'].sum():,}")
print(f"Diferencia frente a las 1.263 coincidencias reportadas por el profesor: "
      f"{int(validas['CoincideConEstimado'].sum()) - 1263:+,}")

if "MontoEstimadoVisible" in entrada_auditada:
    visibilidad = normalizar_codigo(entrada_auditada["MontoEstimadoVisible"])
    no_visibles = visibilidad.eq("NO").sum()
    print(f"Ofertas con monto estimado no visible: {no_visibles:,} de {len(ofertas):,} ({no_visibles / len(ofertas):.1%})")
    display(validas.groupby(["MonedaLicitacion", "VisibilidadEstimado"]).agg(
        Licitaciones=("BrechaMonto", "size"),
        Coincidencias=("CoincideConEstimado", "sum"),
        MedianaBrecha=("BrechaMonto", "median"),
    ))

contexto = [col for col in brechas if col.endswith((
    "_Informada", "_MencionaAdjudicacion", "_MencionaPresupuesto",
))]
if contexto:
    display(validas.groupby("CoincideConEstimado")[contexto].sum().T)
print("Valores extremos por IQR logarítmico en CLP:")
display(ofertas["EsExtremoLogIQR_CLP"].value_counts(dropna=False).to_frame("Ofertas"))

                         Nulos en entrada  Fallos de conversión
Columna                                                        
FechaPublicacion                        0                     0
FechaCierre                             0                     0
FechaAdjudicacion                      91                     0
MontoNetoOferta                         0                     0
MontoTotalOferta                        0                     0
MontoEstimadoLicitacion                 0                     0

                                Nulos al leer  Nulos después de convertir y normalizar  % faltantes                      Regla                                                                       Motivo
NroLicitacion                             0.0                                        0     0.000000        Conservar faltantes  Identifica la licitación. Sin código, la fila queda fuera de la agregación.
MonedaLicitacion                          0.0                                        0     0.000000        Conservar faltantes       Unidad del estimado. Si falta, no puede verificarse la compatibilidad.
MontoEstimadoLicitacion                   0.0                                        0     0.000000        Conservar faltantes        Referencia de comparación. Sin monto válido se excluye la licitación.
MontoEstimadoVisible                      0.0                                        0     0.000000        Conservar faltantes        Distingue visibilidad pública del monto; vacío sig

                                      Licitaciones  Coincidencias  MedianaBrecha
MonedaLicitacion VisibilidadEstimado                                            
CLF              NO                             10              0        -114.58
                 SI                             10              0        -21.595
CLP              NO                           1189            635            0.0
                 SI                           1041            617            0.0
USD              NO                              1              1            0.0
                 SI                              2              1         -0.412
UTM              NO                              4              0         -31.64

CoincideConEstimado                                False True 
BaseEstimacionMontoLicitacion_Informada             1003  1254
BaseEstimacionMontoLicitacion_MencionaAdjudicacion     0     0
BaseEstimacionMontoLicitacion_MencionaPresupuesto    666   907
JustificacionMontoEstimado_Informada                 313   475
JustificacionMontoEstimado_MencionaAdjudicacion        2     1
JustificacionMontoEstimado_MencionaPresupuesto       105   155

                     Ofertas
EsExtremoLogIQR_CLP         
False                  44716
True                     887
<NA>                     138

Columnas de entrada tras el filtro de identificadores: 71
Filas adicionales idénticas: 22
Cobertura por licitación: 2,257 de 2,292 (98.5%)
Todas las comparables
Licitaciones: 2,257
Coincidencias exactas: 1,254
Coincidencias con tolerancia F2: 1,254
Sensibilidad sin ganadoras duplicadas
Licitaciones: 2,253
Coincidencias exactas: 1,254
Coincidencias con tolerancia F2: 1,254
Diferencia frente a las 1.263 coincidencias reportadas por el profesor: -9
Ofertas con monto estimado no visible: 24,895 de 45,741 (54.4%)
Valores extremos por IQR logarítmico en CLP:


## 7. Evidencias para el informe técnico

- Incluye el enlace al commit del notebook y del módulo, la bitácora del pipeline y ejemplos de las clases y de la suscripción del observador.
- Reporta las dimensiones y el resultado de la comparación con **cada** salida de F2.
- Incorpora la tabla de tiempo y memoria e indica el entorno de ejecución. Distingue el costo lineal teórico de los tiempos observados.
- Registra los parámetros ajustados del IQR y explica por qué los dos archivos analíticos tienen unidades diferentes.
- Usa las cifras del bloque 6 de esta versión. Las 502 brechas y los 7.635 atípicos citados en la devolución corresponden al método anterior.
- Antes de entregar, comprueba que el repositorio excluya el CSV original, que el README indique cómo obtener la fuente y dónde colocarla, y que el equipo gestione la eliminación de copias con identificadores que sigan en el historial. Mantén locales las columnas de texto libre. Registra los aportes del equipo con sus propios commits.